This notebook:

1. Reads a file of an MGRS tile
2. Identifies the overlapping OPERA RTC Burst IDs
3. Downloads the metadata for all the overlapping bursts from CMR/ASF
4. Saves this metadata as a parquet file
5. Downloads and organizes this data

The CMRrequests

# Get Metadata for Burst Time Series

In [1]:
from dist_s1_enumerator import get_burst_ids_in_mgrs_tiles, localize_rtc_s1_ts
from dist_s1_enumerator.asf import get_rtc_s1_ts_metadata_by_burst_ids, append_pass_data
from tqdm.auto import tqdm
import pandas as pd
import geopandas as gpd
import concurrent.futures
import warnings
import backoff
from requests.exceptions import HTTPError
from time import sleep
import datetime

/u/duvel-d2/cmarshak/miniforge3/envs/dist-s1-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open('MGRS_tiles.txt') as f:
    mgrs_tile_ids = f.readlines()
mgrs_tile_ids = list(map(lambda x: x.strip(), mgrs_tile_ids))
mgrs_tile_ids[:3]

['33MXU', '20KRV', '20LMK']

In [3]:
burst_ids = get_burst_ids_in_mgrs_tiles(mgrs_tile_ids)
burst_ids[:3], len(burst_ids)

(['T058-123139-IW1', 'T058-123139-IW2', 'T058-123140-IW1'], 5303)

In [5]:
%%time

df_burst_ts = gpd.GeoDataFrame()

@backoff.on_exception(backoff.expo, (HTTPError, ConnectionError), max_tries=10, max_time=60, jitter=backoff.full_jitter)
def get_table_for_burst_id(burst_id: str) -> gpd.GeoDataFrame():
    try:
        df_burst = get_rtc_s1_ts_metadata_by_burst_ids(burst_id)
    except ValueError:
        print(f'Burst id with mixed polarizations:', burst_id)
        df_burst = None
    return df_burst       

with warnings.catch_warnings():
    # Ignore warnings about empty dataframes
    warnings.simplefilter("ignore", category=UserWarning)
    with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
        dfs = list(tqdm(executor.map(get_table_for_burst_id, burst_ids[:]), total=len(burst_ids)))

In [ ]:
dfs_f = list(filter(lambda x: x is not None, dfs))

In [ ]:
dfs_f[0].head()

In [ ]:
df_burst_ts = pd.concat(dfs_f, axis=0)
df_burst_ts.head()

In [7]:
# %%time

# df_burst_ts.to_parquet('burst_ts.parquet', compression='zstd', index=None)

# Serialize Data

In [8]:
df_burst_ts = gpd.read_parquet('burst_ts.parquet')

In [9]:
# This updates metadata to include MGRS info; also for downloading data using dist-s1-enumerator which has column validation
df_burst_ts = append_pass_data(df_burst_ts, mgrs_tile_ids)

# # I am not sure why these are not correctly typed.
df_burst_ts.track_number = df_burst_ts.track_number.astype(int)
df_burst_ts.pass_id = df_burst_ts.pass_id.astype(int)


In [10]:
df_burst_ts.head()

,opera_id,jpl_burst_id,acq_dt,acq_date_for_mgrs_pass,polarizations,track_number,pass_id,url_crosspol,url_copol,geometry,mgrs_tile_id,acq_group_id_within_mgrs_tile,track_token
0,OPERA_L2_RTC-S1_T001-000054-IW1_20220108T18030...,T001-000054-IW1,2022-01-08 18:03:08+00:00,2022-01-08,VV+VH,1,488,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,"POLYGON ((1.54238 9.60217, 2.30972 9.75694, 2....",31PCM,0,1
1,OPERA_L2_RTC-S1_T001-000054-IW1_20220120T18030...,T001-000054-IW1,2022-01-20 18:03:07+00:00,2022-01-20,VV+VH,1,490,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,"POLYGON ((1.54341 9.60227, 2.31075 9.75702, 2....",31PCM,0,1
2,OPERA_L2_RTC-S1_T001-000054-IW1_20220201T18030...,T001-000054-IW1,2022-02-01 18:03:07+00:00,2022-02-01,VV+VH,1,492,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,"POLYGON ((1.5437 9.60213, 2.31102 9.75687, 2.2...",31PCM,0,1
3,OPERA_L2_RTC-S1_T001-000054-IW1_20220213T18030...,T001-000054-IW1,2022-02-13 18:03:07+00:00,2022-02-13,VV+VH,1,494,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,"POLYGON ((1.5426 9.60194, 2.30995 9.7567, 2.27...",31PCM,0,1
4,OPERA_L2_RTC-S1_T001-000054-IW1_20220225T18030...,T001-000054-IW1,2022-02-25 18:03:06+00:00,2022-02-25,VV+VH,1,496,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,"POLYGON ((1.54243 9.60217, 2.30977 9.75694, 2....",31PCM,0,1


In [11]:
# %%time

# df_burst_ts_loc = localize_rtc_s1_ts(df_burst_ts[:], 'out/burst_ts_data', tqdm_enabled=True, max_workers=5)
# df_burst_ts_loc.head()

In [12]:
%%time
now = datetime.datetime.now()
print(now)
while True:
    try:
        df_burst_ts_loc = localize_rtc_s1_ts(df_burst_ts[:], 'out/burst_ts_data', tqdm_enabled=True, max_workers=20)
        break
    except:
        now = datetime.datetime.now()
        sleep(20)
        print(f'retrying... (at {now})')
df_burst_ts_loc.head()

2025-06-25 18:54:39.937325


2025-06-25 19:00:25.881749
retrying... (at 2025-06-25 19:00:25.881749)


2025-06-25 19:38:07.430406
retrying... (at 2025-06-25 19:38:07.430406)


2025-06-25 19:53:59.439151
retrying... (at 2025-06-25 19:53:59.439151)


2025-06-25 20:28:54.207917
retrying... (at 2025-06-25 20:28:54.207917)


...

2025-06-25 23:33:21.055470
retrying... (at 2025-06-25 23:33:21.055470)


2025-06-25 23:36:20.128334
retrying... (at 2025-06-25 23:36:20.128334)


2025-06-26 00:06:18.091543
retrying... (at 2025-06-26 00:06:18.091543)


CPU times: user 1h 10min 6s, sys: 1h 2min 57s, total: 2h 13min 3s
Wall time: 5h 33min 57s


,opera_id,jpl_burst_id,acq_dt,acq_date_for_mgrs_pass,polarizations,track_number,pass_id,url_crosspol,url_copol,geometry,mgrs_tile_id,acq_group_id_within_mgrs_tile,track_token,loc_path_copol,loc_path_crosspol
0,OPERA_L2_RTC-S1_T001-000054-IW1_20220108T18030...,T001-000054-IW1,2022-01-08 18:03:08+00:00,2022-01-08,VV+VH,1,488,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,"POLYGON ((1.54238 9.60217, 2.30972 9.75694, 2....",31PCM,0,1,out/burst_ts_data/31PCM/1/2022-01-08/OPERA_L2_...,out/burst_ts_data/31PCM/1/2022-01-08/OPERA_L2_...
1,OPERA_L2_RTC-S1_T001-000054-IW1_20220120T18030...,T001-000054-IW1,2022-01-20 18:03:07+00:00,2022-01-20,VV+VH,1,490,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,"POLYGON ((1.54341 9.60227, 2.31075 9.75702, 2....",31PCM,0,1,out/burst_ts_data/31PCM/1/2022-01-20/OPERA_L2_...,out/burst_ts_data/31PCM/1/2022-01-20/OPERA_L2_...
2,OPERA_L2_RTC-S1_T001-000054-IW1_20220201T18030...,T001-000054-IW1,2022-02-01 18:03:07+00:00,2022-02-01,VV+VH,1,492,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,"POLYGON ((1.5437 9.60213, 2.31102 9.75687, 2.2...",31PCM,0,1,out/burst_ts_data/31PCM/1/2022-02-01/OPERA_L2_...,out/burst_ts_data/31PCM/1/2022-02-01/OPERA_L2_...
3,OPERA_L2_RTC-S1_T001-000054-IW1_20220213T18030...,T001-000054-IW1,2022-02-13 18:03:07+00:00,2022-02-13,VV+VH,1,494,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,"POLYGON ((1.5426 9.60194, 2.30995 9.7567, 2.27...",31PCM,0,1,out/burst_ts_data/31PCM/1/2022-02-13/OPERA_L2_...,out/burst_ts_data/31PCM/1/2022-02-13/OPERA_L2_...
4,OPERA_L2_RTC-S1_T001-000054-IW1_20220225T18030...,T001-000054-IW1,2022-02-25 18:03:06+00:00,2022-02-25,VV+VH,1,496,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,"POLYGON ((1.54243 9.60217, 2.30977 9.75694, 2....",31PCM,0,1,out/burst_ts_data/31PCM/1/2022-02-25/OPERA_L2_...,out/burst_ts_data/31PCM/1/2022-02-25/OPERA_L2_...


In [14]:
%%time

df_burst_ts_loc.to_parquet('burst_ts_loc.parquet', compression='zstd', index=None)

CPU times: user 1.01 s, sys: 330 ms, total: 1.34 s
Wall time: 1.41 s
